# 01 Preprocessing

**Assignment 2: Explainable Maternal Health Risk Prediction** · KQC7016 Data Analytics

Goals of this notebook:
1. Load the raw UCI Maternal Health Risk dataset and audit its quality.
2. Fix the impossible `HeartRate = 7` sensor-error rows (outlier handling).
3. Remove the 562 exact duplicate rows **before** splitting, to prevent
   train/test leakage (identical rows landing in both sets would inflate accuracy).
4. Encode the ordinal target and produce a **stratified** train/test split.
5. Also produce a *full-dataset* (duplicates kept) split for a later **sensitivity
   analysis** that quantifies the leakage/inflation effect.

Outputs written to `data/`: `maternal_clean.csv`, `train.csv`, `test.csv`,
`train_full.csv`, `test_full.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

raw = pd.read_csv("../dataset/Maternal Health Risk Data Set.csv")
print("Raw shape:", raw.shape)
raw.head()

Raw shape: (1014, 7)


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


## 1. Quality audit

We confirm the three facts that drive every preprocessing decision: there are **no
missing values** (so missing-value handling is trivial), there are **many exact
duplicate rows**, and there are a handful of **physiologically impossible** records.

In [2]:
print("Columns :", list(raw.columns))
print("Dtypes  :\n", raw.dtypes, sep="")
print("\nTotal nulls          :", int(raw.isnull().sum().sum()))
print("Exact duplicate rows :", int(raw.duplicated().sum()))
print("Unique rows          :", raw.drop_duplicates().shape[0])
print("\nClass balance:")
print(raw["RiskLevel"].value_counts())

Columns : ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']
Dtypes  :
Age              int64
SystolicBP       int64
DiastolicBP      int64
BS             float64
BodyTemp       float64
HeartRate        int64
RiskLevel       object
dtype: object

Total nulls          : 0
Exact duplicate rows : 562
Unique rows          : 452

Class balance:
RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64


In [3]:
# Numeric ranges — spot the impossible / extreme values
raw.describe().T[["min", "25%", "50%", "75%", "max"]]

,min,25%,50%,75%,max
Age,10.0,19.0,26.0,39.0,70.0
SystolicBP,70.0,100.0,120.0,120.0,160.0
DiastolicBP,49.0,65.0,80.0,90.0,100.0
BS,6.0,6.9,7.5,8.0,19.0
BodyTemp,98.0,98.0,98.0,98.0,103.0
HeartRate,7.0,70.0,76.0,80.0,90.0


### Outlier check: `HeartRate`

A resting heart rate of **7 bpm** is incompatible with life. These are sensor errors,
not real measurements. The dataset minimum is 7 while the next valid values sit in the
normal 60–90 range, so we drop these rows. Age extremes (10, 70) and `BS = 19` are
clinically plausible (teen pregnancy, older mothers, severe hyperglycaemia) and are
**kept**, noted in the report.

In [4]:
print("HeartRate == 7 rows:", int((raw["HeartRate"] == 7).sum()))
print("HeartRate sorted unique (low end):", sorted(raw["HeartRate"].unique())[:6])
raw[raw["HeartRate"] == 7]

HeartRate == 7 rows: 2
HeartRate sorted unique (low end): [7, 60, 65, 66, 67, 68]


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
499,16,120,75,7.9,98.0,7,low risk
908,16,120,75,7.9,98.0,7,low risk


## 2. Cleaning

Apply the outlier fix to the **full** dataset first, so the *only* difference between the
primary (deduplicated) and sensitivity (full) datasets later is the duplicate rows. Then
encode the ordinal target.

In [5]:
# 2a. Drop impossible HeartRate rows (applies to both primary & sensitivity sets)
clean_full = raw[raw["HeartRate"] != 7].copy()
print(f"After HeartRate fix: {raw.shape[0]} -> {clean_full.shape[0]} rows")

# 2b. Encode ordinal target: low(0) < mid(1) < high(2)
RISK_MAP = {"low risk": 0, "mid risk": 1, "high risk": 2}
clean_full["RiskLevel"] = clean_full["RiskLevel"].str.strip().str.lower()
clean_full["risk_encoded"] = clean_full["RiskLevel"].map(RISK_MAP)
assert clean_full["risk_encoded"].notna().all(), "Unmapped RiskLevel label!"
print("Label map:", RISK_MAP)
clean_full["risk_encoded"].value_counts().sort_index()

After HeartRate fix: 1014 -> 1012 rows
Label map: {'low risk': 0, 'mid risk': 1, 'high risk': 2}


risk_encoded
0    404
1    336
2    272
Name: count, dtype: int64

In [6]:
# 2c. Deduplicate (primary modelling set). Dedup on the original feature+label columns.
feature_cols = ["Age", "SystolicBP", "DiastolicBP", "BS", "BodyTemp", "HeartRate"]
clean_dedup = clean_full.drop_duplicates(subset=feature_cols + ["RiskLevel"]).reset_index(drop=True)
print(f"Deduplicated primary set: {clean_full.shape[0]} -> {clean_dedup.shape[0]} unique rows")
print("\nPrimary set class balance:")
print(clean_dedup["RiskLevel"].value_counts())

Deduplicated primary set: 1012 -> 451 unique rows

Primary set class balance:
RiskLevel
low risk     233
high risk    112
mid risk     106
Name: count, dtype: int64


## 3. Stratified train/test split

Stratify on `RiskLevel` so all three risk classes keep their proportions in both splits.
The split is done **once here** and saved, so every downstream notebook loads the *same*
train/test partition — no accidental re-splitting, no leakage.

- **Primary** split: on the deduplicated set (used for the real model evaluation).
- **Sensitivity** split: on the full duplicate-containing set (used later to show how
  duplicates inflate performance).

In [7]:
def stratified_split(df, tag):
    train, test = train_test_split(
        df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["RiskLevel"]
    )
    print(f"[{tag}] train={train.shape[0]}  test={test.shape[0]}")
    return train.reset_index(drop=True), test.reset_index(drop=True)

train, test = stratified_split(clean_dedup, "primary  ")
train_full, test_full = stratified_split(clean_full, "sensitivity")

[primary  ] train=360  test=91
[sensitivity] train=809  test=203


In [8]:
# Confirm class proportions are preserved in the primary split
prop = pd.DataFrame({
    "train_%": train["RiskLevel"].value_counts(normalize=True).round(3) * 100,
    "test_%":  test["RiskLevel"].value_counts(normalize=True).round(3) * 100,
})
prop

,train_%,test_%
RiskLevel,,
low risk,51.7,51.6
high risk,24.7,25.3
mid risk,23.6,23.1


### Leakage sanity check

Because the primary set is deduplicated on **features + label**, every exact `(vitals, risk)`
record exists only once and lands on a single side of the split, meaning there is **no
inflating leakage** (a test row sharing both its vitals *and* label with a train row).

The check below counts test rows whose six vitals match a train row. In the **full** split
this is huge (these are the exact duplicates that inflate accuracy, the leakage the
deduplication removes). In the **primary** split the residual matches are *not* leakage:
they are the dataset's **conflicting-label patterns** (identical vitals, different risk
label). Those straddle the split and *cap* accuracy rather than inflate it, an irreducible
noise floor we report, not a methodological flaw.

In [9]:
def feature_overlap(train_df, test_df):
    tr = set(map(tuple, train_df[feature_cols].values))
    te = list(map(tuple, test_df[feature_cols].values))
    matched = sum(1 for row in te if row in tr)
    return matched, len(te)

m_p, n_p = feature_overlap(train, test)
m_f, n_f = feature_overlap(train_full, test_full)
print(f"Primary (dedup): {m_p}/{n_p} test rows share vitals with a train row ({m_p/n_p:.1%})")
print("                 -> conflicting-label patterns, NOT inflating leakage")
print(f"Full  (with dup): {m_f}/{n_f} test rows share vitals with a train row ({m_f/n_f:.1%})")
print("                 -> exact-duplicate leakage that dedup removes")

Primary (dedup): 10/91 test rows share vitals with a train row (11.0%)
                 -> conflicting-label patterns, NOT inflating leakage
Full  (with dup): 161/203 test rows share vitals with a train row (79.3%)
                 -> exact-duplicate leakage that dedup removes


## 4. Save outputs

| File | Rows | Used by |
|---|---|---|
| `maternal_clean.csv` | deduplicated, HR-fixed | EDA (02), ARM (04) |
| `train.csv` / `test.csv` | deduplicated split | Classification (03), Explainability (05) |
| `train_full.csv` / `test_full.csv` | full split (dups kept) | Sensitivity analysis (03) |

In [10]:
clean_dedup.to_csv(DATA_DIR / "maternal_clean.csv", index=False)
train.to_csv(DATA_DIR / "train.csv", index=False)
test.to_csv(DATA_DIR / "test.csv", index=False)
train_full.to_csv(DATA_DIR / "train_full.csv", index=False)
test_full.to_csv(DATA_DIR / "test_full.csv", index=False)

for f in ["maternal_clean.csv", "train.csv", "test.csv", "train_full.csv", "test_full.csv"]:
    print(f"saved data/{f}")

saved data/maternal_clean.csv
saved data/train.csv
saved data/test.csv
saved data/train_full.csv
saved data/test_full.csv


**Summary.** Raw 1,014 rows → dropped 2 impossible `HeartRate=7` records (duplicates of
each other) → 1,012 rows; the primary modelling set keeps the **451 unique rows** that remain
after removing the exact `(vitals, risk)` duplicates, split 80/20 stratified by risk class.
Duplicates were removed *before* splitting to avoid train/test leakage; the full-set split is
retained for a sensitivity comparison in notebook 03. The only residual feature overlap in the
primary split comes from conflicting-label patterns (irreducible noise), not leakage.
Next: **02_eda**.